# 12 · Django publishes an order event

**Level:** intermediate · **Time:** 40–55 minutes

**You will:** trace an HTTP request to a queue, process an event in Django, replay a duplicate, and inspect the database effect.


Prerequisites: notebook 00; Docker Compose running RabbitMQ, PostgreSQL, web and worker.
From the repository root run `docker compose up -d --build web worker` and wait for startup.
Use a Python notebook kernel on the host. Celery workers run in Linux containers, including on Windows.
These notebooks call HTTP instead of importing Django ORM into Jupyter's running async event loop.
Cells finish or time out; run top to bottom. New orders are created each run.


## 1. Read the route through the system

`Notebook → HTTP POST → Django transaction → Pika publisher → RabbitMQ queue`

`Management-command consumer → database transaction → acknowledgement`

The publisher and consumer are separate pieces. A successful publish means the broker accepted the
event; it does not mean the order is completed. This lesson uses **direct** mode, which publishes
after the database commits. Lesson 14 closes the remaining publication gap with an outbox.
First check the HTTP service and define bounded helpers for later cells.


In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess
import requests
from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
API = os.getenv("DJANGO_BASE_URL", "http://localhost:8000").rstrip("/")

def get_order(order_id):
    response = requests.get(f"{API}/api/orders/{order_id}/", timeout=5)
    response.raise_for_status()
    return response.json()

def wait_completed(order_id, timeout=30):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        order = get_order(order_id)
        if order["status"] == "completed":
            return order
        time.sleep(0.2)
    raise TimeoutError(f"Order {order_id} is still pending. Inspect worker/consumer logs.")

def manage(*args, check=True, broker_override=None):
    command = ["docker", "compose", "exec", "-T"]
    if broker_override is not None:
        command += ["-e", f"BROKER_URL={broker_override}"]
    command += ["web", "python", "manage.py", *map(str, args)]
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, timeout=60)
    if check and completed.returncode:
        raise RuntimeError(completed.stderr or completed.stdout)
    return completed

response = requests.get(f"{API}/health/", timeout=5)
response.raise_for_status()
print(response.json())


## 2. Inspect the actual application code

Read the `create_order` transaction and `publish_event`. Locate `on_commit`, `confirm_delivery`,
`mandatory=True`, the queue binding, and `delivery_mode=2`. They solve different problems.
The local JSON API intentionally has no authentication and is CSRF-exempt; it is bound to localhost.


In [ ]:
print((ROOT / "django_demo" / "orders" / "views.py").read_text())
print((ROOT / "django_demo" / "orders" / "messaging.py").read_text())


## 3. Create an order

**Predict:** the response is HTTP 201 and the order remains `pending`. The separate event consumer
is not running yet. Django commits the row before publishing, so a consumer cannot beat the commit.
If this returns 503, the response includes the already-saved order identifier: inspect it instead of
blindly retrying POST and creating another order.


In [ ]:
response = requests.post(f"{API}/api/orders/", json={
    "product": "Python messaging workbook", "quantity": 2, "mode": "direct"
}, timeout=30)
print(response.status_code, response.json())
response.raise_for_status()
order = response.json()
assert order["status"] == "pending"
assert order["fulfillments"] == 0


## 4. Observe the waiting event

Open RabbitMQ Management at http://localhost:15672, select virtual host `learning`, then the
`rmqcourse.orders.process` queue. You should see a ready message; dashboard statistics can lag.
Do not use the Management "Get messages" control with automatic acknowledgement, because that
would remove the message before our Django consumer sees it.

Now run the finite consumer. It drains at most 100 messages for at most three seconds. Existing
events from previous runs may also be processed. The output reports the individual database effects.


In [ ]:
result = manage("consume_events", "--max-messages", 100, "--timeout", 3)
print(result.stdout)
completed = get_order(order["id"])
assert completed["status"] == "completed"
assert completed["fulfillments"] == 1
print(completed)


## 5. Replay exactly the same event

An acknowledgement can be lost after a database commit. The broker may then redeliver the message.
Simulate that duplicate explicitly by publishing the same event ID again. A unique event record and
the business update share one database transaction. This protects this database effect; it does not
make delivery exactly once or protect an unrelated external payment/email automatically.


In [ ]:
import pika
from course.rabbit import connect
event = {"schema_version": 1, "event_id": order["event_id"], "order_id": order["id"],
         "product": order["product"], "quantity": order["quantity"]}
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    channel.basic_publish(exchange="rmqcourse.orders", routing_key="order.created",
                          body=json.dumps(event).encode(), mandatory=True,
                          properties=pika.BasicProperties(delivery_mode=2, content_type="application/json",
                                                          message_id=event["event_id"]))
print(manage("consume_events", "--max-messages", 100, "--timeout", 3).stdout)
assert get_order(order["id"])["fulfillments"] == 1
print("Duplicate delivered; database effect still occurred once.")


## 6. Exercise: add an audit subscriber

Create a separate temporary queue bound to `order.*`. Publish the same event again and prove that
both the processing queue and the audit queue receive a copy. Explain why a second consumer on
the **same** processing queue would share work instead of receiving another copy.

Try it before opening the solution cell. Its queue is deleted when the connection closes.


In [ ]:
# Suggested solution: an independent binding creates a second destination.
from course.rabbit import wait_for_message
with connect() as connection:
    channel = connection.channel()
    audit = channel.queue_declare(queue="", exclusive=True).method.queue
    channel.queue_bind(queue=audit, exchange="rmqcourse.orders", routing_key="order.*")
    channel.confirm_delivery()
    channel.basic_publish(exchange="rmqcourse.orders", routing_key="order.created",
                          body=json.dumps(event).encode(), mandatory=True)
    method, properties, body = wait_for_message(channel, audit)
    assert json.loads(body)["event_id"] == event["event_id"]
    channel.basic_ack(method.delivery_tag)
    print("Audit subscriber received:", json.loads(body))
print(manage("consume_events", "--max-messages", 100, "--timeout", 3).stdout)
assert get_order(order["id"])["fulfillments"] == 1


## Checkpoint, cleanup, and troubleshooting

- Explain why publisher confirms cannot establish that `fulfillments` is one.
- Explain why `ProcessedEvent` and the order change must commit atomically.
- A pending order after a confirmed publish usually means the consumer has not run; inspect its output.
- `PRECONDITION_FAILED` can mean an older queue has different declaration arguments. Inspect it before
  changing or deleting it; all Django lessons share this topology.
- Leave the completed rows as demonstration evidence. Temporary audit resources were automatically
  removed. Do not delete the shared Django queue between these lessons. `docker compose down`
  stops the local services and keeps their named data volumes.

Read: [RabbitMQ confirms](https://www.rabbitmq.com/docs/confirms),
[Django transactions](https://docs.djangoproject.com/en/5.2/topics/db/transactions/).
Continue with **13 · Django and Celery**.
